# 02: DMD on 1D Burgers Equation

This notebook uses a simple 1D viscous Burgers equation as CFD/PDE time-series data, then applies the Exact DMD implementation from `src/dmd.py`.

Project structure:

- `src/burgers.py`: generate PDE snapshots `u(x, t)`.
- `src/dmd.py`: implement Exact DMD and reconstruction.
- `notebooks/02_burgers_dmd.ipynb`: run the experiment and plot results.


## 1. Import packages and local code

If imports fail, first check that the selected kernel is `Python (koopman_sciml)` or `PyTorch (pytorch_env)` and that `sys.executable` points to the Anaconda environment with NumPy and Matplotlib.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from burgers import make_burgers_1d
from dmd import exact_dmd, reconstruct, relative_error, snapshot_pair

print("Python:", sys.executable)
print("Project root:", PROJECT_ROOT)


## 2. Generate Burgers snapshots

We solve

$$u_t + u u_x = \nu u_{xx}$$

and store the solution as a snapshot matrix. The DMD convention is:

- each column = one time snapshot,
- `data.shape = (number of spatial points, number of time points)`.

In [ ]:
x, t, data = make_burgers_1d(nx=256, nt=181, t_end=1.5, nu=0.03)

print("x shape:", x.shape)
print("t shape:", t.shape)
print("data shape:", data.shape)


In [ ]:
plt.figure(figsize=(8, 4))
plt.imshow(
    data,
    aspect="auto",
    origin="lower",
    extent=[t[0], t[-1], x[0], x[-1]],
    cmap="turbo",
)
plt.colorbar(label="u(x, t)")
plt.xlabel("time")
plt.ylabel("x")
plt.title("1D viscous Burgers snapshots")
plt.tight_layout()


## 3. Fit Exact DMD

DMD uses adjacent snapshot pairs:

$$X = [x_1, x_2, \dots, x_{m-1}], \quad X' = [x_2, x_3, \dots, x_m].$$

Here we train on the first 70% of the trajectory, then use the learned DMD model to reconstruct and forecast.

In [ ]:
train_steps = int(0.70 * data.shape[1])
train_data = data[:, :train_steps]

X, Xp = snapshot_pair(train_data)
rank = 12
model = exact_dmd(X, Xp, rank=rank)

recon_train = reconstruct(model, train_steps)
pred_all = reconstruct(model, data.shape[1], x0=data[:, 0])

print("DMD rank:", model.rank)
print("training relative error:", relative_error(train_data, recon_train))
print("full-trajectory relative error:", relative_error(data, pred_all))


## 4. Eigenvalue plot

Eigenvalues near the unit circle correspond to slowly decaying or nearly persistent temporal structures. Eigenvalues well inside the unit circle decay faster.

In [ ]:
theta = np.linspace(0.0, 2.0 * np.pi, 400)

plt.figure(figsize=(5, 5))
plt.plot(np.cos(theta), np.sin(theta), "k--", linewidth=1, label="unit circle")
plt.scatter(model.eigvals.real, model.eigvals.imag, s=40)
plt.axhline(0.0, color="0.7", linewidth=1)
plt.axvline(0.0, color="0.7", linewidth=1)
plt.xlabel("Real")
plt.ylabel("Imaginary")
plt.title("DMD eigenvalues")
plt.axis("equal")
plt.legend()
plt.tight_layout()


## 5. Compare true solution and DMD prediction

For a nonlinear PDE, Exact DMD is only a linear approximation in the original state space. This is why Burgers is a useful bridge from DMD to Koopman/EDMD.

In [ ]:
indices = [0, train_steps // 2, train_steps - 1, data.shape[1] - 1]

plt.figure(figsize=(9, 6))
for i, idx in enumerate(indices, start=1):
    plt.subplot(2, 2, i)
    plt.plot(x, data[:, idx], label="true", linewidth=2)
    plt.plot(x, pred_all[:, idx], "--", label="DMD", linewidth=2)
    plt.title(f"t = {t[idx]:.2f}")
    plt.xlabel("x")
    plt.ylabel("u")
    if i == 1:
        plt.legend()

plt.tight_layout()


## 6. Error versus rank

This is a useful figure for your report: increasing rank gives the linear model more degrees of freedom, but it still cannot fully remove the nonlinear-modeling error.

In [ ]:
ranks = [2, 4, 6, 8, 12, 16, 24, 32]
train_errors = []
full_errors = []

for r in ranks:
    m = exact_dmd(X, Xp, rank=r)
    train_pred = reconstruct(m, train_steps)
    full_pred = reconstruct(m, data.shape[1], x0=data[:, 0])
    train_errors.append(relative_error(train_data, train_pred))
    full_errors.append(relative_error(data, full_pred))

plt.figure(figsize=(6, 4))
plt.semilogy(ranks, train_errors, "o-", label="train reconstruction")
plt.semilogy(ranks, full_errors, "s-", label="full trajectory")
plt.xlabel("DMD rank")
plt.ylabel("relative error")
plt.title("DMD error versus rank")
plt.grid(True, which="both", alpha=0.3)
plt.legend()
plt.tight_layout()


## 7. What this proves

This notebook is not just another CFD script. It gives you a clean transition:

1. Burgers equation provides nonlinear PDE snapshot data.
2. Exact DMD learns the best low-rank linear time-evolution model from data.
3. The mismatch on nonlinear dynamics motivates Koopman and EDMD, where the state is lifted to a richer observable space.